# 2: Taxonomy-Kraken

In [ ]:
# reinstall new version of kraken 
conda create -n kraken2_v2.17.1 kraken2==2.17.1  
# https://github.com/DerrickWood/kraken2/blob/master/docs/MANUAL.markdown
# https://bioconda.github.io/recipes/kraken2/README.html

In [1]:
# Nikea has tried a couple different dbs - see here: https://github.com/nikeaulrich/DR_SCTLD/blob/main/DR_kraken_abundances.ipynb
# Now she is proceeding with PlusPF db
# For now i will use this before trying to build custom db

## Batch Scripts - Kraken

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

# have to wait on PSTR and PAST since I am redoing their assembly steps...
SPP_LIST="MCAV|MMEA|NEG|ORBI"
DBNAME="/datasets/bio/kraken2/PlusPF"
LISTPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
SAMPLELIST="filtered_sample_groups.txt"


# classify each set of paired end reads against Pracken database
while read -r SAMPLEID SPECIE GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    
    # only continue with selected species 
    if [[ ! "$SPECIE" =~ ^($SPP_LIST)$ ]]; then
        echo "Skipping $SAMPLEID ($SPECIE) - not in target list."
        continue
    fi
    echo "Processing ${SPECIE}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly/final_filtered"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    mkdir -p "$OUTDIR"
    
    kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "${SPP_LIST}" processed successfully."

# also do PSTR
SPP="PSTR"
while IFS= read -r SAMPLEID SPECIE GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    
    # only continue with selected species 
    if [[ ! "$SPECIE" = "$SPP" ]]; then
        continue
    fi
    echo "Processing ${SPECIE}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly2/final_filtered"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    
    kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_TASK --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "$SPP" processed successfully."


conda deactivate

# JOB-ID: 
# bash script file name: kraken2-mostspp

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 4:00:00  # Job time limit
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-neg-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

DBNAME="/datasets/bio/kraken2/PlusPF"
SPP="NEG"

FINAL_SAMPLES="7_3_Neg"
READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
# mkdir -p "$OUTDIR"

for SAMPLEID in $FINAL_SAMPLES
do
kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 \
    --report-zero-counts \
    --paired $READS/"${SAMPLEID}_R1.tagged_filter_ready.fastq.gz" \
             $READS/"${SAMPLEID}_R2.tagged_filter_ready.fastq.gz" > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done

echo "Kraken2: All samples in "${SPP}" processed successfully."

In [ ]:
# run past 

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 18:00:00  # Job time limit
#SBATCH --array=1-53%10
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-past-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

SPP="PAST"
DBNAME="/datasets/bio/kraken2/PlusPF"
LISTPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}"
SAMPLELIST="spp_samples"

INPUT_FILE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" input.txt)

while read -r SAMPLEID GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    echo "Processing ${SPP}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly/final_filtered2"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    
    kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "$SPP" processed successfully."


conda deactivate

# JOB-ID: 
# bash script file name: kraken2-past

In [ ]:
# try array jobs instead - using PAST
# failed using 1 and 4 hours per sample..trying 8 

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH --cpus-per-task=16
#SBATCH --mem=250G
#SBATCH -t 01:00:00
#SBATCH --array=1-53%10
#SBATCH --mail-type=ALL
#SBATCH --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-past-%A_%a.out

module load conda/latest
conda activate kraken2_v2.17.1

SPP="PAST"

# array job
# sample ID list
SAMPLE_LIST="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/spp_samples"
SAMPLEID=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$SAMPLE_LIST")

# paths for kraken
READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered2"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
DBNAME="/datasets/bio/kraken2/PlusPF"

R1="${READS}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz"
R2="${READS}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz"

echo "Processing task ID $SLURM_ARRAY_TASK_ID: Sample $SAMPLEID"
kraken2 --db "$DBNAME" \
        --threads "$SLURM_CPUS_PER_TASK" \
        --report "$OUTDIR/${SAMPLEID}.kreport2" \
        --report-zero-counts \
        --paired "$R1" "$R2" \
        > "$OUTDIR/${SAMPLEID}.kraken2"
if [ $? -eq 0 ]; then
    echo "kraken2 completed successfully for sample: $SAMPLEID"
else
    echo "kraken2 encountered an error for sample: $SAMPLEID"
    exit 1
fi

conda deactivate

# JOB-ID: 61265954
# bash script file name: kraken2-past

## Bracken

In [ ]:
# re-install latest udpate
# install in latest kraken2 env (unity has kraken2 as a module but mine is more recently updated)

conda activate kraken2_v2.17.1 
conda install -c bioconda bracken # Bracken v3.01

In [ ]:
module load conda/latest
conda activate kraken2_v2.17.1 

BRACKEN_PATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken"
mkdir -p $BRACKEN_PATH

# build bracken db 
DBNAME="/datasets/bio/kraken2/PlusPF"
THREADS=20
KMER_LEN=35
READ_LEN=150
BRACKEN_BUILD_PATH=/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/bash_scripts
# Generate bracken database - just do once
# $BRACKEN_BUILD_PATH/bracken-build.txt -d ${DBNAME} -t ${THREADS} -k ${KMER_LEN} -l ${READ_LEN} 
bracken-build -d ${DBNAME} -t ${THREADS} -k ${KMER_LEN} -l ${READ_LEN} 
# ${KRAKEN_DB}`  = location of the built Kraken 1/Kraken 2/KrakenUniq database
# ${THREADS}`    = number of threads to use with Kraken and the Bracken scripts
# ${KMER_LEN}`   = length of kmer used to build the Kraken database 
#     Kraken 1/KrakenUniq default kmer length = 31
#     Kraken 2 default kmer length = 35
#     Default set in the script is 35. 
# ${READ_LEN}`   = the read length of your data e.g., if you are using 100 bp reads, set it to `100`. 
#need taxo.k2d, hash.k2d, and opts.k2d. in kraken2 database

# Abundance Estimation
KRAKENFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
#ls $FILEPATH | find *.kreport2 > kraken_list.txt
#cat kraken_list.txt| sed 's/.kreport2/''/' > SAMPLEID
SCRIPTPATH=/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/bash_scripts/KrakenTools-master/DiversityTools
#$BRACKEN_BUILD_PATH/bash install_bracken.sh
level='G'